In [1]:
import numpy as np

In [2]:
# Load your solute XYZ positions
def load_xyz(filename):
    atoms, coords = [], []
    with open(filename, 'r') as f:
        lines = f.readlines()[2:]
        for line in lines:
            parts = line.split()
            atoms.append(parts[0])
            coords.append([float(x) for x in parts[1:4]])
    return atoms, np.array(coords)

# Write combined XYZ file
def write_xyz(filename, atoms, coords):
    with open(filename, 'w') as f:
        f.write(f"{len(atoms)}\nGenerated water box\n")
        for atom, coord in zip(atoms, coords):
            f.write(f"{atom} {coord[0]:.4f} {coord[1]:.4f} {coord[2]:.4f}\n")

# Create water molecule relative positions (TIP3P-like)
def create_water_coords(center):
    O = center
    H1 = center + np.array([0.9572, 0.0000, 0.0000])
    H2 = center + np.array([-0.2390, 0.9270, 0.0000])
    return [("O", O), ("H", H1), ("H", H2)]




In [3]:
# Parameters
xbox_size = 14.0  # Angstrom
ybox_size = 10.0  # Angstrom
zbox_size = 12.0  # Angstrom
spacing = 3.0    # Distance between water molecules
water_atoms, water_coords = [], []

# Generate grid of water molecules
for x in np.arange(-xbox_size/2, xbox_size/2, spacing):
    for y in np.arange(-ybox_size/2, ybox_size/2, spacing):
        for z in np.arange(-zbox_size/2, zbox_size/2, spacing):
            mol = create_water_coords(np.array([x, y, z]))
            for atom, coord in mol:
                water_atoms.append(atom)
                water_coords.append(coord)

water_coords = np.array(water_coords)

# Load solute
solute_atoms, solute_coords = load_xyz("molecule.xyz")

# Combine solute and water
combined_atoms = solute_atoms + water_atoms
combined_coords = np.vstack((solute_coords, water_coords))

# Write final XYZ file
write_xyz("solvated_system.xyz", combined_atoms, combined_coords)

print("Water molecules added around solute and written to solvated_system.xyz")

Water molecules added around solute and written to solvated_system.xyz


In [ ]:
#############222222222222
import numpy as np

def load_xyz(filename):
    atoms, coords = [], []
    with open(filename, 'r') as f:
        lines = f.readlines()[2:]
        for line in lines:
            parts = line.split()
            atoms.append(parts[0])
            coords.append([float(x) for x in parts[1:4]])
    return atoms, np.array(coords)

def write_xyz(filename, atoms, coords):
    with open(filename, 'w') as f:
        f.write(f"{len(atoms)}\nCombined system\n")
        for atom, coord in zip(atoms, coords):
            f.write(f"{atom} {coord[0]:.4f} {coord[1]:.4f} {coord[2]:.4f}\n")

def distance(p1, p2):
    return np.linalg.norm(p1 - p2)

# Load solute and water box
solute_atoms, solute_coords = load_xyz("molecule.xyz")
water_atoms, water_coords = load_xyz("water.xyz")

# Keep water molecules within a shell distance (e.g., 12 Å) from solute center
solute_center = np.mean(solute_coords, axis=0)
max_distance = 1.0

selected_atoms, selected_coords = [], []
for atom, coord in zip(water_atoms, water_coords):
    if distance(coord, solute_center) <= max_distance:
        selected_atoms.append(atom)
        selected_coords.append(coord)

# Combine solute and selected water
combined_atoms = solute_atoms + selected_atoms
combined_coords = np.vstack((solute_coords, selected_coords))

# Write final combined XYZ file
write_xyz("solvated_system.xyz", combined_atoms, combined_coords)

print(f"{len(selected_atoms)} water atoms added around solute.")